# G07 · A2 — Full-corpus Tesseract OCR baseline

**Scope:** runs the repo's **default** pipeline (`ingest.load_pages` → `layout.detect`
[projection] → `ocr.transcribe` [tesseract]) over the **entire ~1,034-page** Pierce (1890)
corpus, using your own uploaded PDF. This is the pretrained-baseline evidence for A2 form
Section 3 ("options compared") — not a fine-tune, not the held-out label set (that's the
separate `kaggle_heldout_ocr.ipynb`).

**Runtime warning:** Tesseract runs page-by-page, region-by-region, on CPU. With ~1,034
pages this is realistically **multiple hours**, not minutes. The OCR loop below is
**resumable** — each page's output is written to disk as soon as it's done, so a rerun
skips finished pages instead of starting over. For a long unattended run, use Kaggle's
**Save Version → Save & Run All (commit)** rather than an interactive session, so it keeps
running if you close the tab.

**Quick test first:** set `PAGE_LIMIT` in the config cell to something small (e.g. `20`) to
sanity-check the whole pipeline in a couple of minutes before committing to the full run.


In [ ]:
# ── 1. Environment ────────────────────────────────────────────────────────────
import os, sys, subprocess, time, json

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

%pip install -q "pymupdf>=1.25.5,<1.26" "opencv-python-headless>=4.10,<5.0" "pytesseract>=0.3.13,<0.4" "pydantic>=2.7,<3.0" "pydantic-settings>=2.2,<3.0" "pyyaml>=6.0,<7.0"

if subprocess.run(["which", "tesseract"], capture_output=True).returncode != 0:
    sh("apt-get -qq update > /dev/null 2>&1 || true")
    sh("apt-get -qq install -y tesseract-ocr > /dev/null 2>&1")

REPO_URL = "https://github.com/smammahdi/doc-agent-G07.git"
PIN = "45c3fc3"  # tip of a2/pierce-kb-foundation
if not os.path.exists("/kaggle/working/repo"):
    sh(f"git clone -q {REPO_URL} /kaggle/working/repo")
sh(f"cd /kaggle/working/repo && git checkout -q {PIN} && git log --oneline -1")
sys.path.insert(0, "/kaggle/working/repo/src")

import fitz, cv2, pytesseract  # noqa: E402
TESSERACT_VERSION = subprocess.run(["tesseract", "--version"], capture_output=True, text=True).stdout.splitlines()[0]
print("python     :", sys.version.split()[0])
print("tesseract  :", TESSERACT_VERSION)
print("pymupdf    :", getattr(fitz, "__version__", None) or fitz.version[0])


In [ ]:
# ── 2. PDF: your own uploaded copy (no download; soft integrity check only) ──
from pathlib import Path
import hashlib

PDF = Path("/kaggle/input/datasets/kmazd1110/dl-peoples-common-sense-med-advisor/"
            "EN_The-Peoples-Common-Sense-Medical-Adviser.pdf")
if not PDF.exists():
    raise FileNotFoundError(
        f"{PDF} not found. Check the exact mount path in the Kaggle 'Input' panel "
        "(right sidebar) — Kaggle sometimes mounts at /kaggle/input/<dataset-slug>/... "
        "without the extra 'datasets/' segment — and update PDF above to match."
    )

# scripts/get_data.sh pins the 1890 IA edition (item peoplescommonsen00pier) to this hash.
# Several near-identical-looking editions exist on IA (1875/1876/1890) — this is a fast,
# non-blocking check that you uploaded the same edition the rest of the repo assumes.
EXPECTED_BYTES = 65311598
EXPECTED_SHA = "841b1feb55ff0aff5735c3aeb308eb52e217f91ae55c5d34e21feb6a640c8896"

def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

size = PDF.stat().st_size
digest = sha256(PDF)
doc_probe = fitz.open(str(PDF))
n_pages = len(doc_probe)
doc_probe.close()
print(f"file       : {PDF}")
print(f"size       : {size:,} bytes  (expected {EXPECTED_BYTES:,})")
print(f"sha256     : {digest}")
print(f"pages      : {n_pages}  (provenance record expects 1,034)")
if size == EXPECTED_BYTES and digest == EXPECTED_SHA:
    print("MATCH: byte-identical to the pinned 1890 IA edition (peoplescommonsen00pier).")
else:
    print("WARNING: does not match the pinned 1890 edition's size/hash. If this is a "
          "different scan of the same book, the page count/results may not line up with "
          "the rest of the repo (page IDs, provenance record). Not blocking — continuing.")


In [ ]:
# ── 3. Config: repo defaults, only the source path/output dir overridden ──────
from doc_agent import config as config_mod  # noqa: E402

PAGE_LIMIT = None  # set e.g. 20 for a fast smoke test before the full run

cfg = config_mod.load("/kaggle/working/repo/configs/config.yaml")
cfg["ingest"]["source_pdf"] = str(PDF)
cfg["ingest"]["output_dir"] = "/kaggle/working/data/processed/pages"
cfg["ingest"]["page_limit"] = PAGE_LIMIT
OUT = Path("/kaggle/working/out")
(OUT / "ocr_text").mkdir(parents=True, exist_ok=True)

print("ingest:", cfg["ingest"])
print("layout:", cfg["layout"])
print("ocr   :", cfg["ocr"])


In [ ]:
# ── 4. Render every page (300 DPI JPEG q80, the repo default) ────────────────
from doc_agent.ingest import loader  # noqa: E402

t0 = time.time()
pages = loader.load_pages(cfg)
print(f"rendered/verified {len(pages)} pages in {time.time() - t0:.1f}s "
      f"(cached pages are skipped on rerun)")


In [ ]:
# ── 5. Resumable page-by-page OCR: projection layout + Tesseract ─────────────
from doc_agent.vision import layout as layout_mod  # noqa: E402
from doc_agent.vision import ocr as ocr_mod        # noqa: E402

TEXT_DIR = OUT / "ocr_text"
CHUNKS_PATH = OUT / "chunks.jsonl"
STATS_PATH = OUT / "stats.jsonl"
done = {p.stem for p in TEXT_DIR.glob("*.txt")}
print(f"already done: {len(done)}/{len(pages)} pages (resuming)")

reader = ocr_mod.Reader(cfg)  # built once; reused across pages (cheap for tesseract mode)
t_start = time.time()
n_done_this_run = 0
with open(CHUNKS_PATH, "a", encoding="utf-8") as chunks_f, open(STATS_PATH, "a", encoding="utf-8") as stats_f:
    for i, page in enumerate(pages):
        if page.id in done:
            continue
        t_page = time.time()
        regions = layout_mod.detect([page], cfg)
        texts = [reader.transcribe_region(r) for r in regions]
        page_text = "\n\n".join(t for t in texts if t)

        tmp = TEXT_DIR / f".{page.id}.txt.tmp"
        tmp.write_text(page_text, encoding="utf-8")
        tmp.rename(TEXT_DIR / f"{page.id}.txt")  # atomic

        for j, (region, text) in enumerate(zip(regions, texts, strict=True)):
            row = dict(id=f"{page.id}:r{j:04d}", doc_id=page.doc_id, text=text,
                       page_ids=[page.id], score=0.0)
            chunks_f.write(json.dumps(row, ensure_ascii=False) + "\n")
        chunks_f.flush()

        words = sum(len(t.split()) for t in texts)
        stat = dict(page_id=page.id, regions=len(regions), words=words,
                    chars=len(page_text), elapsed_s=round(time.time() - t_page, 2))
        stats_f.write(json.dumps(stat) + "\n")
        stats_f.flush()

        n_done_this_run += 1
        if n_done_this_run % 25 == 0:
            rate = n_done_this_run / (time.time() - t_start)
            remaining = len(pages) - len(done) - n_done_this_run
            eta_min = remaining / rate / 60 if rate > 0 else float("inf")
            print(f"[{i + 1}/{len(pages)}] {page.id}  {rate:.2f} pages/s  ETA {eta_min:.0f} min")

print(f"\nfinished this run: {n_done_this_run} pages "
      f"({time.time() - t_start:.1f}s). Rerun this cell to resume if interrupted.")


In [ ]:
# ── 6. Aggregate stats ─────────────────────────────────────────────────────────
import csv

stats = [json.loads(line) for line in open(STATS_PATH, encoding="utf-8")]
assert len(stats) == len(pages), (
    f"{len(stats)} stat rows vs {len(pages)} pages — rerun cell 5 until this matches "
    "(full-corpus run isn't finished yet)."
)

total_words = sum(s["words"] for s in stats)
total_chars = sum(s["chars"] for s in stats)
total_regions = sum(s["regions"] for s in stats)
zero_word = sum(1 for s in stats if s["words"] == 0)
total_elapsed_h = sum(s["elapsed_s"] for s in stats) / 3600

summary = dict(
    engine=TESSERACT_VERSION, layout="projection (repo default)", commit=PIN,
    source_pdf=str(PDF), source_sha256=digest, page_count=len(pages),
    total_regions=total_regions, total_words=total_words, total_chars=total_chars,
    zero_word_pages=zero_word, total_ocr_hours=round(total_elapsed_h, 2),
)
(OUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

with open(OUT / "stats.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["page_id", "regions", "words", "chars", "elapsed_s"])
    w.writeheader()
    w.writerows(stats)

print(json.dumps(summary, indent=2))


## Sanity check

Eyeball a couple of pages before trusting the numbers above.


In [ ]:
# ── 7. Eyeball a few pages: rendered image vs extracted text ──────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

sample_ids = [pages[10].id, pages[len(pages) // 2].id, pages[-10].id]
for pid in sample_ids:
    img_path = Path(cfg["ingest"]["output_dir"]) / cfg["ingest"]["doc_id"] / \
        f'{cfg["ingest"]["dpi"]}dpi-{cfg["ingest"]["format"]}-q{cfg["ingest"]["quality"]}' / f"{pid}.jpg"
    txt = (TEXT_DIR / f"{pid}.txt").read_text(encoding="utf-8")
    img = mpimg.imread(str(img_path))
    h, w = img.shape[:2]
    plt.figure(figsize=(6, 6 * h / w)); plt.imshow(img); plt.axis("off")
    plt.title(pid); plt.show()
    print(f"── {pid} extracted text (first 400 chars) ──")
    print(txt[:400])
    print()


In [ ]:
# ── 8. Package for download ────────────────────────────────────────────────────
import shutil

zip_path = shutil.make_archive("/kaggle/working/tesseract_baseline_output", "zip", str(OUT))
print("DONE. Download from the right panel → Output:")
print("  tesseract_baseline_output.zip")
print("  contains: ocr_text/pNNNN.txt (per page), chunks.jsonl (region-level, Chunk schema),")
print("            stats.csv, stats.jsonl, summary.json")
print()
print(json.dumps(summary, indent=2))
